https://pub.towardsai.net/parquet-best-practices-the-art-of-filtering-d729357e441d

In [2]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import time
import os

In [3]:
def get_first_parquet_from_path(path):
    for (dir_path, _, files) in os.walk(path):
        for f in files:
            if f.endswith(".parquet"):
                first_pq_path = os.path.join(dir_path, f)
                return first_pq_path
path = 'APPLICATIONS_PROCESSED'
first_pq = get_first_parquet_from_path(path)
pq.read_schema(first_pq)

ID: int64
CODE_GENDER: string
FLAG_OWN_CAR: string
FLAG_OWN_REALTY: string
CNT_CHILDREN: int64
AMT_INCOME_TOTAL: double
NAME_EDUCATION_TYPE: string
NAME_FAMILY_STATUS: string
NAME_HOUSING_TYPE: string
DAYS_EMPLOYED: int64
FLAG_MOBIL: bool
FLAG_WORK_PHONE: bool
FLAG_PHONE: bool
FLAG_EMAIL: bool
OCCUPATION_TYPE: string
CNT_FAM_MEMBERS: double
MONTH_INCOME_TOTAL: double
AGE: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [{"name": null, "field_n' + 2567

In [5]:
cols=['ID', 'CNT_CHILDREN', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL']
df=pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols)
df.head()

,ID,CNT_CHILDREN,DAYS_EMPLOYED,AMT_INCOME_TOTAL
0,3,0,-3051,270000.0
1,4,0,-3051,270000.0
2,5,0,-3051,270000.0
3,6,0,-3051,270000.0
4,13,0,-1194,135000.0


In [6]:
cols=['ID', 'CNT_CHILDREN', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL']
start_time = time.time()
df=pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols)
print(f'{np.round(time.time() - start_time, 2)} seconds with column pruning')

start_time = time.time()
df_slow = pd.read_parquet('APPLICATIONS_PROCESSED')[cols]
print(f'{np.round(time.time() - start_time, 2)} seconds for the slow version of loading all and filtering after')


0.15 seconds with column pruning
1.25 seconds for the slow version of loading all and filtering after


In [7]:
def get_all_partitions(path):
    partitions = {}
    i = 0
    for (_, partitions_layer, _) in os.walk(path):
        if len(partitions_layer)>0:
            key = partitions_layer[0].split('=')[0]
            partitions[key] = sorted([partitions_layer[i].split('=')[1] for i in range(len(partitions_layer))])
        else:
            break
    return partitions
ps = get_all_partitions('APPLICATIONS_PROCESSED')
ps.keys(), ps.values()

(dict_keys(['NAME_INCOME_TYPE']),
 dict_values([['Commercial%20associate', 'Pensioner', 'State%20servant', 'Student', 'Working']]))

In [8]:
cols = ['ID', 'CNT_CHILDREN', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL']
filter_part = [('NAME_INCOME_TYPE', 'in', ('Pensioner', 'Working'))]
start_time = time.time()
df_pp = pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols, filters=filter_part)
print(f'{np.round(time.time() - start_time, 2)} seconds with columns pruning and partition pruning')

start_time = time.time()
df_pp_slow = pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols+['NAME_INCOME_TYPE'])
df_pp_slow = df_pp_slow[df_pp_slow['NAME_INCOME_TYPE'].isin(['Pensioner', 'Working'])]
print(f'{np.round(time.time() - start_time, 2)} seconds with columns pruning + filtering afterwards')

0.06 seconds with columns pruning and partition pruning
0.27 seconds with columns pruning + filtering afterwards


= or ==, !=, <, >, <=, >=, in and not in

In [9]:
filter_part = [[('NAME_INCOME_TYPE', 'in', ('Pensioner', 'working'))],[('CODE_GENDER','=','F')]]
df_pp2 = pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols, filters=filter_part)
df_pp2.head()

,ID,CNT_CHILDREN,DAYS_EMPLOYED,AMT_INCOME_TOTAL
0,3,0,-3051,270000.0
1,4,0,-3051,270000.0
2,5,0,-3051,270000.0
3,6,0,-3051,270000.0
4,66,2,-1773,126000.0


Predicate pushdown is a technique used to filter data at the storage layer before it is read into memory. 

In [10]:
cols = ['ID', 'CNT_CHILDREN', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL']
filter_part = [('AMT_INCOME_TOTAL','>', 250000)]
start_time = time.time()
df_pp3 = pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols, filters=filter_part)
print(f'{np.round(time.time() - start_time, 2)} seconds with Column pruning and Predicate Pushdown')

0.09 seconds with Column pruning and Predicate Pushdown


In [11]:
print(df_pp.shape)
print(df_pp3.shape)

(6031940, 4)
(1569420, 4)


In [12]:
cols = ['ID', 'CNT_CHILDREN', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL']
filter_part = [('AMT_INCOME_TOTAL','>', 250000)]
start_time = time.time()
df_slow = pd.read_parquet('APPLICATIONS_PROCESSED', columns=cols)
df_slow = df_slow.loc[(df_slow['AMT_INCOME_TOTAL']>250000), cols]
print(f'{np.round(time.time() - start_time, 2)} seconds without profiting from predicate pushdown')

0.23 seconds without profiting from predicate pushdown
